In [ ]:
# FFmpeg
!apt-get update -qq
!apt-get install -y ffmpeg

# Diarization
!pip install diarize

# Audio + ASR dependencies
!pip install -U \
    "transformers>=5.4.0" \
    accelerate \
    huggingface_hub \
    soundfile \
    librosa \
    sentencepiece \
    protobuf \
    demucs

!pip install torchvision==0.23.0

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:6.1.1-3ubuntu5).
0 upgraded, 0 newly installed, 0 to remove and 140 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.9/887.9 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 MB 7.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 84.3 MB/s eta 0:00:00
  Attempting uninstall: torchvision
    Found existing installation: torchvision 0.26.0+cu128
    Uninstalling torchvision-0.26.0+cu128:
      Successfully uninstalled torchvision-0.26.0+cu128


In [ ]:
import importlib.metadata as metadata
import torch
import torchvision
import torchaudio
import transformers

from diarize import diarize

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

for package in [
    "torchvision",
    "torchaudio",
    "transformers",
    "demucs",
    "diarize"
]:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: Not installed")

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("All imports successful!")

PyTorch: 2.8.0+cu128
CUDA available: True
torchvision: 0.23.0
torchaudio: 2.8.0
transformers: 5.18.0
demucs: 4.1.0
diarize: 0.1.2
GPU: Tesla T4
All imports successful!


In [ ]:
from google.colab import files
from pathlib import Path
import subprocess

print("Is your file Audio or Video?")
file_type = input().strip().lower()

if file_type == "audio":

    uploaded = files.upload()

    if not uploaded:
        raise ValueError("No audio file uploaded.")

    audio_name = next(iter(uploaded.keys()))
    audio_path = f"/content/{audio_name}"

    print("Audio uploaded successfully!")
    print(f"Audio: {audio_path}")

elif file_type == "video":

    uploaded = files.upload()

    if not uploaded:
        raise ValueError("No video file uploaded.")

    video_name = next(iter(uploaded.keys()))
    video_path = f"/content/{video_name}"

    audio_path = "/content/audio.wav"

    command = [
        "ffmpeg",
        "-hide_banner",
        "-loglevel", "error",
        "-y",
        "-i", video_path,
        "-vn",
        "-ac", "1",
        "-ar", "16000",
        "-c:a", "pcm_s16le",
        audio_path
    ]

    subprocess.run(command, check=True)

    if not Path(audio_path).exists():
        raise FileNotFoundError(audio_path)

    print("Audio extracted successfully!")
    print(f"Video: {video_path}")
    print(f"Audio: {audio_path}")

else:
    print("The file type is not supported. Please enter Audio or Video.")

Saving YTDown.com_YouTube_Media_YXMyeGUURsQ_E-26-Prompt-Engineering-using-LangChain-️-Memory-Conclusion-Ch-7-8-8_001_720p.mp4 to YTDown.com_YouTube_Media_YXMyeGUURsQ_E-26-Prompt-Engineering-using-LangChain-️-Memory-Conclusion-Ch-7-8-8_001_720p.mp4
Audio extracted successfully!
Video: /content/YTDown.com_YouTube_Media_YXMyeGUURsQ_E-26-Prompt-Engineering-using-LangChain-️-Memory-Conclusion-Ch-7-8-8_001_720p.mp4
Audio: /content/audio.wav


In [ ]:
import torchaudio
from IPython.display import Audio, display

waveform, sample_rate = torchaudio.load(audio_path)

display(
    Audio(
        waveform.squeeze(0).numpy(),
        rate=sample_rate
    )
)

/usr/local/lib/python3.13/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be r

In [ ]:
import torch
import torchaudio

from demucs.pretrained import get_model
from demucs.apply import apply_model

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

# Load Demucs model
demucs_model = get_model(name="htdemucs_ft")
demucs_model.to(device)
demucs_model.eval()

# Load audio
wav, sr = torchaudio.load(audio_path)

# Demucs expects stereo audio
if wav.shape[0] == 1:
    wav = wav.repeat(2, 1)
elif wav.shape[0] > 2:
    wav = wav[:2]

# Resample to model's expected sample rate
target_sr = demucs_model.samplerate

if sr != target_sr:
    wav = torchaudio.functional.resample(
        wav,
        orig_freq=sr,
        new_freq=target_sr
    )

wav = wav.to(device)

# Demucs normalization
ref = wav.mean(dim=0)
wav = wav - ref.mean()

mean = wav.mean()
std = wav.std().clamp_min(1e-8)

wav = (wav - mean) / std

with torch.inference_mode():
    sources = apply_model(
        demucs_model,
        wav.unsqueeze(0),
        device=device,
        progress=True
    )[0]

# Restore original scale
sources = sources * std + mean

sources_dict = dict(zip(demucs_model.sources, sources))

# Extract vocals
vocals = sources_dict["vocals"].cpu()

# Convert vocals to mono
vocals_mono = vocals.mean(dim=0, keepdim=True)

# Resample to 16 kHz
vocals_16k = torchaudio.functional.resample(
    vocals_mono,
    orig_freq=target_sr,
    new_freq=16000
)

vocals_path = "/content/vocals.wav"

torchaudio.save(
    vocals_path,
    vocals_16k,
    16000
)

print("Vocals saved:", vocals_path)

# Listen to isolated vocals
vocals_check, vocals_sr = torchaudio.load(vocals_path)

display(
    Audio(
        vocals_check.squeeze(0).numpy(),
        rate=vocals_sr
    )
)

Device: cuda


htdemucs_ft.yaml:   0%|          | 0.00/149 [00:00<?, ?B/s]

f7e0c4bc.safetensors: reconstructing file:   0%|          |  0.00B / 84.0MB            

f7e0c4bc.safetensors: downloading bytes:           |  0.00B            

d12395a8.safetensors: reconstructing file:   0%|          |  0.00B / 84.0MB            

...

92cfc3b6.safetensors: downloading bytes:           |  0.00B            

04573f0d.safetensors: reconstructing file:   0%|          |  0.00B / 84.0MB            

04573f0d.safetensors: downloading bytes:           |  0.00B            

100%|████████████████████████████████████████████████████████████████████████| 105.3/105.3 [00:05<00:00, 20.58seconds/s]
100%|████████████████████████████████████████████████████████████████████████| 105.3/105.3 [00:04<00:00, 25.79seconds/s]
100%|████████████████████████████████████████████████████████████████████████| 105.3/105.3 [00:03<00:00, 27.10seconds/s]
100%|████████████████████████████████████████████████████████████████████████| 105.3/105.3 [00:03<00:00, 27.00seconds/s]
/usr/local/lib/python3.13/dist-packages/torchaudio/_backend/utils.py:337: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.save_with_torchcodec` under the hood. Some parameters like format, encoding, bits_per_sample, buffer_size, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's encoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.encoders.AudioEncoder
  warnings.warn(
/usr/local/lib/python3.13/

Vocals saved: /content/vocals.wav


In [ ]:
USE_DEMUCS = True

AUDIO_FILE = (
    "/content/vocals.wav"
    if USE_DEMUCS
    else "/content/audio.wav"
)

OUTPUT_FILE = "/content/output.json"

print("ASR input:", AUDIO_FILE)

ASR input: /content/vocals.wav


In [ ]:
import torch

from transformers import (
    AutoProcessor,
    CohereAsrForConditionalGeneration
)

device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_NAME = "CohereLabs/cohere-transcribe-arabic-07-2026"

processor = AutoProcessor.from_pretrained(
    MODEL_NAME
)

model = CohereAsrForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=(
        torch.float16
        if device == "cuda"
        else torch.float32
    )
).to(device)

model.eval()

print("Cohere ASR model loaded successfully.")

processor_config.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/131 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/3.42k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.82M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.13GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/2150 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

Cohere ASR model loaded successfully.


In [ ]:
import numpy as np
import torchaudio

waveform, sample_rate = torchaudio.load(AUDIO_FILE)

if waveform.shape[0] > 1:
    waveform = waveform.mean(dim=0, keepdim=True)

if sample_rate != 16000:
    waveform = torchaudio.functional.resample(
        waveform,
        sample_rate,
        16000
    )
    sample_rate = 16000

audio = waveform.squeeze(0).numpy().astype(np.float32)

duration = len(audio) / sample_rate

print("Audio file:", AUDIO_FILE)
print("Sample rate:", sample_rate)
print("Duration:", round(duration, 2), "seconds")

Audio file: /content/vocals.wav
Sample rate: 16000
Duration: 99.61 seconds


/usr/local/lib/python3.13/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be r

In [ ]:
# Force diarization to use exactly one speaker
diarization_output = diarize(
    AUDIO_FILE,
    num_speakers=1
)

diarization_segments = [
    {
        "start": float(segment.start),
        "end": float(segment.end),
        "speaker": str(segment.speaker),
    }
    for segment in diarization_output.segments
]

if not diarization_segments:
    raise RuntimeError(
        "Diarization returned no segments."
    )

print("Number of speakers:", diarization_output.num_speakers)
print("Number of segments:", len(diarization_segments))

for segment in diarization_segments:
    print(
        f"[{segment['start']:.2f}s → {segment['end']:.2f}s] "
        f"{segment['speaker']}"
    )

# Verify that the requested speaker count was returned
if diarization_output.num_speakers != 1:
    raise RuntimeError(
        f"Expected 1 speaker, got "
        f"{diarization_output.num_speakers}"
    )

/usr/local/lib/python3.13/dist-packages/silero_vad/utils_vad.py:202: UserWarning: torchaudio.sox_effects.sox_effects.apply_effects_file has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  return torchaudio.sox_effects.apply_effects_file(path, effects=effects)


voxceleb_resnet34_LM.onnx: 100%|██████████| 25.3M/25.3M [00:04<00:00, 5.85MB/s]


Number of speakers: 1
Number of segments: 7
[1.00s → 11.80s] SPEAKER_00
[12.60s → 37.50s] SPEAKER_00
[38.30s → 52.60s] SPEAKER_00
[54.00s → 64.90s] SPEAKER_00
[66.00s → 72.60s] SPEAKER_00
[73.70s → 95.20s] SPEAKER_00
[96.10s → 97.60s] SPEAKER_00


In [ ]:
def transcribe_segment(start, end):
    start_sample = max(
        0,
        int(start * sample_rate)
    )

    end_sample = min(
        len(audio),
        int(end * sample_rate)
    )

    if end_sample <= start_sample:
        return ""

    segment_audio = audio[start_sample:end_sample]

    inputs = processor(
        audio=segment_audio,
        sampling_rate=sample_rate,
        return_tensors="pt",
        language="ar"
    )

    audio_chunk_index = inputs.get("audio_chunk_index")

    processed_inputs = {}

    for key, value in inputs.items():
        if isinstance(value, torch.Tensor):

            if key == "audio_chunk_index":
                processed_inputs[key] = value.to(device)

            elif key == "input_features":
                processed_inputs[key] = value.to(
                    device=device,
                    dtype=model.dtype
                )

            else:
                processed_inputs[key] = value.to(device)

        else:
            processed_inputs[key] = value

    with torch.inference_mode():
        outputs = model.generate(
            **processed_inputs,
            max_new_tokens=256
        )

    decoded = processor.decode(
        outputs,
        skip_special_tokens=True,
        audio_chunk_index=audio_chunk_index,
        language="ar"
    )

    if isinstance(decoded, list):
        return " ".join(
            item.strip()
            for item in decoded
            if item and item.strip()
        )

    return decoded.strip()

In [ ]:
import json

results = []

for i, segment in enumerate(
    diarization_segments,
    start=1
):
    start = segment["start"]
    end = segment["end"]
    # speaker = segment["speaker"]

    segment_duration = end - start

    # Skip extremely short segments
    if segment_duration < 0.5:
        continue

    print(
        f"[{i}/{len(diarization_segments)}] "
        f"{speaker}: {start:.2f}s - {end:.2f}s"
    )

    try:
        text = transcribe_segment(start, end)

    except Exception as e:
        print("Transcription error:", repr(e))
        text = ""

    results.append({
        # "speaker": speaker,
        "start": round(start, 3),
        "end": round(end, 3),
        "text": text
    })

    print("Text:", text)
    print("-" * 60)


output = {
    "audio_file": AUDIO_FILE,
    "sample_rate": sample_rate,
    "duration": round(duration, 3),
    "diarization_enabled": True,
    "diarization_model": "diarize",
    # "num_speakers": int(diarization_output.num_speakers),
    "segments": results
}

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(
        output,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved:", OUTPUT_FILE)
print("Total transcribed segments:", len(results))

[1/7] SPEAKER_00: 1.00s - 11.80s
Text: الدرس ده ربما يكون أطول درس في السلسلة، ولكن هو أداك كمية معلومات أعتقد كثيرة جداً عن فكرة ال Memory، ووضحلك من Under the hood
------------------------------------------------------------
[2/7] SPEAKER_00: 12.60s - 37.50s
Text: ايه اللي بيحصل في الباك جراوند من وراك؟ مش مجرد انك بتستخدم الميموري وتستسهل وانك تبعت رساله صغيره انك تستهلك كل هذا العدد من التوكينز، لازم تكون حذر جدا في استخدامك للميموري، لكن ما تنكرش ان الميموري تكنيك هنا مناسب جدا لكتير جدا من اليوز كيسز او الحالات اللي انت ممكن تشتغل عليها في الابليكيشن بتاعك، بدانا بالـ Conversation Buffer
------------------------------------------------------------
[3/7] SPEAKER_00: 38.30s - 52.60s
Text: ببساطة جداً إنه دائماً بيشيل حاجة بتحصل كل اللي حصل في الماضي ويرميه استخدمنا طريقة أفضل شوية وهي الـ conversation buffer window memory أننا بحدد له limit معين من الـ history ياخده معاه ويتجاهل الباقي
------------------------------------------------------------
[4/7] SPEAKER_00: 54.00s - 64.90s
Te